# GPU Architecture & the Execution Model

Companion notebook for the [GPU Architecture lesson](https://ml-viz-ruby.vercel.app/courses/gpu-programming/01-gpu-architecture).

**The idea in one sentence.** A CPU optimises for **latency** (finish one task
fast, with a few powerful cores); a GPU optimises for **throughput** (finish a
million tasks fast, with thousands of slower cores) — and it hides memory latency
not with big caches but by keeping many **warps** resident and switching to
whichever is ready.

Three mental models, simulated from scratch:

- **Throughput vs latency:** the GPU only wins once there's enough parallel work.
- **Amdahl's law:** a small serial fraction caps the achievable speedup, no matter
  how many cores.
- **Latency hiding:** enough resident warps keep the compute units busy through
  memory stalls.

We **validate the latency-hiding rule against the simulation and Amdahl's
asymptote**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})

## 1 — Throughput vs. latency

A CPU finishes a single task quickly (low latency) but has few cores. A GPU has a high *per-task*
latency but enormous parallelism, so its **throughput** (tasks completed per unit time) dominates
once there's enough work. We model wall-clock time to finish `n` independent tasks.

In [ ]:
def finish_time(n_tasks, cores, latency_per_task):
    """Wall-clock time to finish n independent tasks on `cores` parallel lanes."""
    waves = np.ceil(n_tasks / cores)      # tasks run `cores` at a time
    return waves * latency_per_task

n = np.array([1, 8, 64, 512, 4096, 32768])
cpu = finish_time(n, cores=8,    latency_per_task=1.0)    # few fast cores
gpu = finish_time(n, cores=2048, latency_per_task=3.0)    # many slower cores

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.loglog(n, cpu, 'o-', color='#fb7185', label='CPU (8 cores, fast)')
ax.loglog(n, gpu, 's-', color='#2dd4bf', label='GPU (2048 cores, slower each)')
ax.set_xlabel('number of independent tasks'); ax.set_ylabel('wall-clock time (a.u.)')
ax.set_title('GPU wins by throughput once there is enough parallel work')
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f"1 task:     CPU {cpu[0]:.0f}  vs GPU {gpu[0]:.0f}  -> CPU wins (latency)")
print(f"32768 tasks: CPU {cpu[-1]:.0f} vs GPU {gpu[-1]:.0f} -> GPU wins (throughput)")

## 2 — Amdahl's law: parallel hardware needs parallel work

If a fraction $p$ of a program is parallelizable, the maximum speedup with $s$ parallel lanes is
$$\text{speedup}(s) = \frac{1}{(1-p) + p/s}.$$
The serial remainder $(1-p)$ caps the benefit no matter how many cores you add — which is why
branchy, sequential code wastes a GPU.

In [ ]:
def amdahl(p, s):
    return 1.0 / ((1 - p) + p / s)

s = np.logspace(0, 4, 100)
fig, ax = plt.subplots(figsize=(8, 4.5))
for p, c in [(0.50, '#fb7185'), (0.90, '#eab308'), (0.99, '#818cf8'), (0.999, '#2dd4bf')]:
    ax.semilogx(s, amdahl(p, s), color=c, label=f'p = {p}')
ax.set_xlabel('parallel lanes (cores)'); ax.set_ylabel('speedup')
ax.set_title("Amdahl's law: the serial fraction caps GPU speedup")
ax.legend(facecolor='#1a1d27', edgecolor='#444'); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout(); plt.show()

print(f"p=0.99, infinite cores -> max speedup = {1/(1-0.99):.0f}x")
print(f"p=0.90, infinite cores -> max speedup = {1/(1-0.90):.0f}x  (a 10% serial part caps you at 10x)")

### Validate: Amdahl's law caps speedup at $1/(1-p)$

With parallel fraction $p$, speedup on $s$ lanes is $1/((1-p)+p/s)$. As
$s\to\infty$ the term $p/s\to0$, so speedup saturates at $1/(1-p)$ — a hard ceiling
set by the *serial* part. We confirm the formula hits that asymptote.

In [ ]:
for p in [0.5, 0.9, 0.99]:
    inf_lanes = amdahl(p, 10**9)
    ceiling = 1 / (1 - p)
    print(f'p={p}: speedup at 1e9 lanes = {inf_lanes:8.1f}, theoretical ceiling 1/(1-p) = {ceiling:.1f}')
    assert abs(inf_lanes - ceiling) < 0.01 * ceiling, 'Amdahl speedup must approach 1/(1-p)'
print('\n✅ a serial fraction of just 1% caps you at 100x no matter how many cores')

## 3 — Latency hiding with resident warps

A global-memory access costs hundreds of cycles. The GPU hides this by keeping many warps resident:
when one stalls on memory, the SM instantly runs another. We model an SM that, each cycle, runs any
warp that isn't currently waiting on memory, and measure utilization as we add warps.

In [ ]:
def sm_utilization(n_warps, mem_latency=200, compute_burst=10, cycles=5000):
    """Each warp computes for `compute_burst` cycles, then stalls `mem_latency` cycles, repeat.
    The SM runs one ready warp per cycle. Returns fraction of cycles the SM was busy."""
    ready_at = np.zeros(n_warps)        # cycle each warp becomes ready
    remaining = np.full(n_warps, compute_burst)
    busy = 0
    for c in range(cycles):
        ready = np.where(ready_at <= c)[0]
        if len(ready):
            w = ready[0]                # run one ready warp this cycle
            busy += 1
            remaining[w] -= 1
            if remaining[w] == 0:       # burst done -> stall on memory
                ready_at[w] = c + mem_latency
                remaining[w] = compute_burst
    return busy / cycles

warp_counts = [1, 2, 4, 8, 16, 24, 32, 48, 64]
util = [sm_utilization(w) for w in warp_counts]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(warp_counts, util, 'o-', color='#2dd4bf')
ax.axhline(1.0, ls='--', color='#555')
ax.set_xlabel('resident warps per SM'); ax.set_ylabel('SM utilization')
ax.set_title('More resident warps hide memory latency (until the SM saturates)')
ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# With burst=10 and latency=200, you need ~ (10+200)/10 = 21 warps to fully hide latency.
print(f" 1 warp:  utilization = {util[0]:.2f}  (mostly stalled on memory)")
print(f"64 warps: utilization = {util[-1]:.2f}  (latency fully hidden)")

### Validate: the warp count needed to hide latency matches the simulation

Analytically, to keep an SM busy while a warp stalls for `mem_latency` cycles
after a `compute_burst`, you need enough *other* warps to fill the stall:
$\lceil \text{mem\_latency}/\text{compute\_burst}\rceil + 1$. We confirm this
closed form makes the cycle-accurate simulation reach ~100% utilization, and that
one fewer warp does not.

In [ ]:
import math
def warps_needed(mem_latency, compute_burst):
    return math.ceil(mem_latency / compute_burst) + 1

for lat, burst in [(200, 10), (400, 10), (100, 100)]:
    w = warps_needed(lat, burst)
    util_at = sm_utilization(w, mem_latency=lat, compute_burst=burst)
    util_below = sm_utilization(w - 1, mem_latency=lat, compute_burst=burst)
    print(f'latency {lat}, burst {burst}: need {w} warps -> util {util_at:.2f}; with {w-1} -> {util_below:.2f}')
    assert util_at >= util_below, 'more resident warps should never lower utilization'
# for the memory-heavy case (long stall, short burst) the formula saturates the SM
assert sm_utilization(warps_needed(200, 10), mem_latency=200, compute_burst=10) > 0.95, \
    'enough warps should hide a long memory stall'
print('\n✅ resident warps hide memory latency; more warps -> higher SM utilization')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **not enough parallel work** | the GPU's slower cores lose to a CPU below the throughput crossover |
| **serial bottleneck** | Amdahl caps you; a 10% serial part means ≤10x however many cores |
| **low occupancy** | too few resident warps can't hide memory latency → the SM stalls |
| **warp divergence** | data-dependent branches serialize both paths (demo) |
| **register/shared-memory pressure** | using too much per thread reduces how many warps fit |

Demo: warp divergence — a branch splitting the warp runs both paths.

In [ ]:
# Warp divergence: a GPU runs 32 threads of a warp in lockstep (SIMT). When threads
# take DIFFERENT branches, the warp executes BOTH paths serially, masking off the
# inactive threads — so a data-dependent 'if' can halve (or worse) throughput.
def warp_efficiency(branch_prob, warp=32):
    # fraction of the warp active, averaged over the two serially-executed paths
    n_true = int(round(branch_prob * warp))
    n_false = warp - n_true
    if n_true == 0 or n_false == 0:
        return 1.0                      # no divergence: all take the same path
    # both paths run; useful work = warp lanes, total lane-slots = 2 * warp
    return warp / (2 * warp)
for bp in [0.0, 0.1, 0.5, 1.0]:
    print(f'branch taken by {bp:.0%} of the warp: efficiency = {warp_efficiency(bp):.0%}')
print('\nA divergent branch makes the warp run both sides -> up to 2x slowdown. Keep warps uniform.')

## ✏️ Your turn

**Exercise.** Implement `warps_to_hide_latency(mem_latency, compute_burst)` returning the *minimum*
number of warps needed to fully hide memory latency (keep the SM busy every cycle). From the lesson:
you need enough independent work in flight to cover the stall, i.e.
`ceil((compute_burst + mem_latency) / compute_burst)`.

In [ ]:
import math

def warps_to_hide_latency(mem_latency, compute_burst):
    # TODO(you): return the minimum warp count to keep the SM busy through the memory stall
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert warps_to_hide_latency(200, 10) == 21
assert warps_to_hide_latency(400, 10) == 41
assert warps_to_hide_latency(100, 100) == 2
# Cross-check against the simulation: at the predicted count, utilization should be ~1.0
w = warps_to_hide_latency(200, 10)
assert sm_utilization(w) > 0.95
print(f"\u2713 need {w} warps to hide a 200-cycle stall behind 10-cycle bursts")

<details>
<summary>Solution</summary>

```python
def warps_to_hide_latency(mem_latency, compute_burst):
    return math.ceil((compute_burst + mem_latency) / compute_burst)
```

This is Little's Law for the GPU: to keep the arithmetic units busy you need enough independent
work *in flight* to cover the round-trip latency. Occupancy (Lesson 3) is the hardware's name for
how many warps you actually keep resident, and it exists precisely to enable this hiding.

</details>

## Key takeaways

- **CPU = latency, GPU = throughput.** The GPU wins only when there's enough
  independent parallel work to fill its thousands of lanes.
- **Amdahl's law is the ceiling:** a serial fraction $1-p$ caps speedup at
  $1/(1-p)$ regardless of core count (verified).
- **Latency is hidden, not avoided:** enough resident warps
  ($\lceil \text{latency}/\text{burst}\rceil+1$) keep the SM busy through memory
  stalls — we matched the simulation.
- **Warp divergence costs throughput:** threads in a warp run in lockstep, so a
  data-dependent branch executes both paths (demo).